In [ ]:
import numpy as np
import pandas as pd
import random
import re
import os
from sklearn.model_selection import train_test_split

In [ ]:
#Load the synthetic dataset dataframes
df = pd.read_csv("# FILEPATH TO SYNTHETIC DATASET CSV")
print(len(df))
df = df.fillna('')
df.head()

In [ ]:
#Save the code for refinement and then perform analysis on the saved code

secure_code = df["Secure Code"].tolist()


#For the refined data security analysis
more_secure_code = df["More Secure Code"].tolist()


final_secure_code = []
count = 0
for i in range(0,len(secure_code)):
    if(more_secure_code[i]=="Nothing" or len(more_secure_code[i])<=0):
        final_secure_code.append(secure_code[i])
    else:
        count+=1
        final_secure_code.append(more_secure_code[i])


secure_code = final_secure_code
df["Secure Code"] = final_secure_code

for idx,item in enumerate(secure_code):
    print(idx)
    f = open(f"refinement/code_{idx}.py","w")
    f.write(item)
    f.close()

for idx,item in enumerate(secure_code):
    bandit_result = os.popen(f"bandit refinement/code_{idx}.py").read()
    f = open(f"refinement/code_{idx}.txt","w")
    f.write(bandit_result)
    f.close()

codeql_result = os.popen(f"./codeql_processing.sh refinement").read()
print("Analysis Complete!")


In [ ]:
#Analyzing and extracting the secure code

def extract_bandit_issues(size_of_dataset):
    issues = []
    file_idx = []

    for i in range(0,len(secure_code)):
        f = open(f"refinement/code_{i}.txt")
        text = f.read()
        start = text.find("Test results:")
        end = text.find("Code scanned:")
        text = text[start+len("Test results:"):end]
        if "No issues identified" in text:
            issues.append("Nothing")
        else:
            file_idx.append(i)
            issues.append(text)
    
    return issues,file_idx

def extract_codeql_issues(size_of_dataset):
    issues = []
    file_idx = []

    f = open("refinement/codeql_analysis.csv","r")
    codeql_info = f.read().splitlines()
    filenames = [f"code_{i}.py" for i in range(size_of_dataset)]
    issues = []
    for item in filenames:
        relevant_issues = [i for i in codeql_info if item in i]
        if(len(relevant_issues)==0):
            issues.append("Nothing")
        else:
            file_idx.append(i)
            issues.append("\n".join(relevant_issues))
    
    return issues,file_idx


secure_code = df["Secure Code"]

bandit_issues,file_idx_bandit = extract_bandit_issues(len(secure_code))
codeql_issues,file_idx_codeql = extract_codeql_issues(len(secure_code))


markers = [0 for i in range(len(secure_code))]
for idx,item in enumerate(bandit_issues):
        if item!="Nothing":
            markers[idx]=1
    
for idx,item in enumerate(codeql_issues):
    if item!="Nothing":
        markers[idx]=1

idx = []

for i in range(0,len(markers)):
    if markers[i] == 1:
        idx.append(i)

idx = set(idx)

print(sum(markers))

df["Bandit Feedback"] = bandit_issues
df["Codeql Feedback"] = codeql_issues

df.to_csv("# FILEPATH TO SYNTHETIC DATASET WITH REFINEMENT FEEDBACK CSV",index=False)